# Visualisation of EEG & Sleep scoring

## Import recordings

Load packages

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import scipy
from scipy import signal
import os
from ephyviewer import mkQApp, MainViewer, TraceViewer
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, InMemoryAnalogSignalSource, EpochEncoder,TimeFreqViewer,AnalogSignalSourceWithScatter, EpochEncoder_ABC
from matplotlib import cm
from matplotlib.colors import to_hex
import re
import mne
import pandas as pd
import numpy as np
import csv
from collections import defaultdict
import ast
import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import string
%matplotlib widget

Choose .edf file

In [ ]:
file= 'AGAL06-E_06032017' #AGAL06-E_06032017 #GUNO03-E_24042018 #AZSA10-E_12112018

Load STS file & scoring infos

In [ ]:
sts_file = f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}.sts"

def extract_printable(binary_data):
    return ''.join(chr(b) for b in binary_data if 32 <= b <= 126 or b in (10, 13))  # keep letters, numbers, space, newline

with open(sts_file, "rb") as f:
    binary_data = f.read()
text_data = extract_printable(binary_data)
keep_words = ["BStage N1", "BStage N2", "BStage N3", "BREM", "BWAKE", "BUnstaged"]
clean_text = re.sub(r"[^\x20-\x7E\n]", "", text_data)
matches = []
lines = clean_text.splitlines()
i = 0
while i < len(clean_text):
    for w in keep_words:
        if clean_text.startswith(w, i):
            matches.append(w[1:])
            i += len(w)
            break
    else:
        i += 1  # move forward if no match
print(f'{len(matches)} vigilance states found')

if "Advanced Artifact Rejection (1=ON, 0=OFF)" in clean_text:
    montagetype = "EPI"
elif "CReformatedMontageSLEEP" in clean_text:
    montagetype = "Sleep"
else:
    montagetype = "custum"
print(f'Montage type detected: {montagetype}')

Start of vigilance state scoring

In [ ]:
# Path to your text file
file_path = f"C:/Users/Manip2/Documents/EpiKid/Lag_VigScoring.txt"
target_name = file 
lag = None
with open(file_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # skip empty lines
        if ";" in line:
            name, number = line.split(";", 1)
            name = name.strip()
            number = number.strip()
            if name == target_name:
                try:
                    lag = float(number)
                except ValueError:
                    print(f"Error: Number for {name} is invalid: {number}")
                break  # stop after finding the first match

if lag is not None:
    print(f"Scoring started at {lag} s")
else:
    print(f"{target_name} not found in the file")
    lag = 0


Load and clean corrupted edf file (no montage)

In [ ]:
dpath = f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}.edf"
folder_base = Path(dpath) 

raw = mne.io.read_raw_edf(
    dpath,
    preload=True,
    verbose="error", 
    encoding='latin1'
)

raw.set_meas_date(None)
raw.set_annotations(mne.Annotations([], [], []))
raw.rename_channels(
    lambda ch: ch[:16].replace(" ", "_")
)
raw.pick_types(
    eeg=True,
    eog=True,
    emg=True,
    ecg=True,
    misc=True
)

data = raw.get_data()
info = raw.info.copy()

raw_clean = mne.io.RawArray(data, info)
mne.export.export_raw(
    f'{Path(dpath).parent}/{Path(dpath).name[:-4]}_repaired.edf',
    raw_clean,
    fmt="edf",
    physical_range=(-1000, 1000),
    overwrite=True
)
# reload to check
data = mne.io.read_raw_edf(f'{Path(dpath).parent}/{Path(dpath).name[:-4]}_repaired.edf', preload=False)
print(data)

Load metadata of edf

In [ ]:
raw_data = data.get_data() #data = mne.io.read_raw_edf(dpath, encoding='latin1')

info = data.info
channels = data.ch_names
timestamps = data.times
duration = data.duration
samplerate = data.info.get('sfreq')  # in Hz

print(data.info.get('meas_date'))
data.info.get('subject_info')
lowpass=data.info.get('lowpass')
highpass=data.info.get('highpass')

combined = raw_data.T
print(f'{round(np.shape(raw_data)[1]/samplerate/60/60,3)}h recording if sampled at {samplerate}Hz ({duration} seconds)')
print(f'{np.shape(raw_data)[0]} channels found: {channels}')

Save scoring for EphyViewer in .csv file

In [ ]:
SleepScoring= pd.DataFrame(matches, columns=['label'])
SleepScoring['time'] = list(range(int(lag), len(SleepScoring) * 30, 30))
SleepScoring['duration'] = 30
AutoSleepScoring_filename = os.path.join(Path(f'{Path(sts_file).parent}'), f'EphyViewer_Scoring_{file}.csv')
SleepScoring.to_csv(AutoSleepScoring_filename, index=False)

Spikes=pd.DataFrame(columns=['label', 'time', 'duration'])
Spikes_filename = os.path.join(Path(f'{Path(sts_file).parent}'), f'EphyViewer_Spikes_{file}.csv')
Spikes.to_csv(Spikes_filename, index=False)


Save scoring for Snooz in .tsv file

In [ ]:
def sec_to_hms(sec):
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{int(h)}:{int(m):02d}:{int(s):02d}"

import pandas as pd

def fill_unstaged_to_length(df, recording_length, epoch=30):
    df = df.sort_values("time").reset_index(drop=True)
    filled = []
    first_start = df.loc[0, "time"]
    if first_start > 0:
        filled.append({
            "label": "Unstaged",
            "time": 0,
            "duration": first_start
        })
    for i, r in df.iterrows():
        filled.append(r.to_dict())
        end_t = r["time"] + r["duration"]
        if i < len(df) - 1:
            next_t = df.loc[i + 1, "time"]
        else:
            next_t = recording_length
        if next_t > end_t:
            gap = next_t - end_t
            t = end_t
            while gap > 0:
                d = min(epoch, gap)
                filled.append({
                    "label": "Unstaged",
                    "time": t,
                    "duration": d
                })
                t += d
                gap -= d
    return pd.DataFrame(filled)

df_filled = fill_unstaged_to_length(SleepScoring, duration, epoch=30)

stage_map = {
    "WAKE": 0,
    "Stage N1": 1,
    "Stage N2": 2,
    "Stage N3": 3,
    "REM": 5,
    "Unstaged": 9,
}

out_df = pd.DataFrame({
    "group": ["stage"] * len(df_filled),
    "name": df_filled["label"].map(stage_map),
    "start_sec": df_filled["time"].astype(int),
    "duration_sec": df_filled["duration"].astype(int),
    "channels": ["[]"] * len(df_filled),
})

# Safety check for unmapped labels
if out_df["name"].isna().any():
    bad = df_filled.loc[out_df["name"].isna(), "label"].unique()
    raise ValueError(f"Unmapped labels found: {bad}")

# Save to TSV
out_df.to_csv(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}_repaired_scoring.tsv", sep="\t", index=False)

Create & save a reapaired .edf with montage (for Snooz & SpikeDet)

In [ ]:
montage = 'Fp2-F4,F4-C4,C4-P4,P4-O2,Fp2-F8,F8-T4,T4-TB2,TB2-A2,A2-T6,T6-O2,Fp1-F3,F3-C3,C3-P3,P3-O1,Fp1-F7,F7-T3,T3-TB1,TB1-A1,A1-T5,T5-O1,Fz-Cz,Cz-Pz,EMG1+,EMG2+'
montage = 'Fp2-F4,F4-C4,C4-P4,P4-O2,Fp2-F8,F8-T4,T4-TB2,T6-O2,Fp1-F3,F3-C3,C3-P3,P3-O1,Fp1-F7,F7-T3,T3-TB1,T5-O1,Fz-Cz,Cz-Pz,EMG1+,EMG2+'
channels_clean = [ch.replace('EEG ', '') for ch in channels]
channels_clean = [ch.replace('EEG_', '') for ch in channels_clean]

pairs = montage.split(',')
montage_eeg=[]
montage_name=[]
for pair in pairs:
    if pair.count('-')==0:
        ch1 = pair
        idx1 = channels_clean.index(ch1)
        eeg=combined[:,idx1]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)
    else:        
        ch1, ch2 = pair.split('-')
        idx1 = channels_clean.index(ch1)
        idx2 = channels_clean.index(ch2)
        eeg=combined[:,idx2]-combined[:,idx1]        
        montage_eeg = eeg[:,np.newaxis] if len(montage_eeg) == 0 else np.append(montage_eeg, eeg[:,np.newaxis], axis=1)
        montage_name.append(pair)
        
montage_eegT = montage_eeg.T

info = mne.create_info(
    ch_names=montage_name,
    sfreq=samplerate,
    ch_types = ['eeg'] * (len(montage_name) - 2) + ['emg', 'emg']
)
raw_clean = mne.io.RawArray(montage_eegT, info)


outpath = Path(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}_repaired_montage.edf")
mne.export.export_raw(
    outpath,
    raw_clean,
    fmt="edf",
    physical_range=(-1000, 1000),
    overwrite=True
)

# Reload to check
data_check = mne.io.read_raw_edf(outpath, preload=False)

Do montage for Ephyviewer (zscore & filter)

In [ ]:
pairs = montage.split(',')
montage_eeg=[]
montage_name=[]
for pair in pairs:
    if pair.count('-')==0:
        ch1 = pair
        idx1 = channels_clean.index(ch1)
        eeg=combined[:,idx1]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)
    else:        
        ch1, ch2 = pair.split('-')
        idx1 = channels_clean.index(ch1)
        idx2 = channels_clean.index(ch2)
        eeg=combined[:,idx2]-combined[:,idx1]
        nyq = samplerate / 2
        f_lowcut = 0.5
        f_hicut = 50.0 #70
        Wn = [f_lowcut/nyq, f_hicut/nyq]
        sos = signal.butter(6, Wn, btype='band', output='sos')
        eeg = signal.sosfiltfilt(sos, eeg)
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        #montage_eeg = (eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, (eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)
eeg=[]
eeg_name=[]
for ch in channels_clean:
    idx1 = channels_clean.index(ch)
    eeg1=combined[:,idx1]
    eeg = zscore(eeg1[:,np.newaxis]) if len(eeg) == 0 else np.append(eeg, zscore(eeg1[:,np.newaxis]), axis=1)
    eeg_name.append(ch)

montage_name_eeg=montage_name[:-2]

## Display signals & vigilance states

Check all channels (no montage)

In [ ]:
app = mkQApp()
winlen = 10 # default window length in sec
t_start = 0.

win = MainViewer(debug=False, show_auto_scale=True)
view1 = TraceViewer.from_numpy(eeg, samplerate, t_start, 'Signals', channel_names=eeg_name)

cmap = cm.Spectral
values = np.linspace(0, 1, np.array(eeg).shape[1])
colormap = [to_hex(rgb) for rgb in cmap(values)]
for idx in range(np.array(eeg).shape[1]): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx]

view1.params['display_labels'] = True
view1.params['scale_mode'] = 'same_for_all'
view1.auto_scale()
view1.params['xsize'] = winlen
win.add_view(view1)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)

#Run
win.show()
app.exec()


With sleep scoring (montage)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(sts_file).parent}/EphyViewer_Scoring_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source =InMemoryAnalogSignalSource(montage_eeg, samplerate, t_start, channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'
view1.params['vline_color'] = "#FFFFFF00"

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#8b8b8b"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.params['vline_color'] = "#FFFFFF00"

view2.controls.hide()



# FFT
view3 = TimeFreqViewer(source=source, name='FFT')

view3.params['show_axis'] = True
view3.params['timefreq', 'f_start'] = 1
view3.params['timefreq', 'f_stop'] = 50
view3.params['timefreq', 'deltafreq'] = 1 #interval in Hz
view3.params['xsize'] = winlen
for idx, ch in enumerate(montage_name):
    """
    if ch == 'EMG': 
        view3.by_channel_params[f'ch{idx}', 'visible'] = False
    else:        
        view3.by_channel_params[f'ch{idx}', 'clim'] = 1
        view3.by_channel_params[f'ch{idx}', 'visible'] = True
    """
    if ch == 'C3-P3': 
        view3.by_channel_params[f'ch{idx}', 'clim'] = 1
        view3.by_channel_params[f'ch{idx}', 'visible'] = True
    else: 
        view3.by_channel_params[f'ch{idx}', 'visible'] = False



win.add_view(view1)
win.add_view(view2)
#win.add_view(view3)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'

## Annotate spikes as epoch on montage

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
spike_dur = .01 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(sts_file).parent}/EphyViewer_Scoring_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

Spikes_filename = f'{Path(sts_file).parent}/EphyViewer_Spikes_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch2 = CsvEpochSource(Spikes_filename, montage_name)


#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
ch_names = [f"{s} ({string.ascii_uppercase[i]})" for i, s in enumerate(montage_name)]
source =InMemoryAnalogSignalSource(montage_eeg, samplerate, t_start, channel_names=ch_names)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = "#FFFFFF"
view1.params['label_fill_color'] = '#FFFFFF'
view1.params['vline_color'] = "#000000"

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()


#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#8b8b8b"

view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = "#FFFFFF"
view2.params['vline_color'] = "#000000"

view2.params['view_mode'] = 'flat'
view2.controls.hide()



#create a viewer for the encoder itself
view4 = EpochEncoder_ABC(source=source_epoch2, name='Spikes')
view4.params['xsize'] = winlen
view4.params['new_epoch_step'] = spike_dur
view4.params['background_color'] = '#FFFFFF'
view4.params['label_fill_color'] = "#FFFFFF"
view4.params['vline_color'] = "#000000"

colormap = ["#FF0000"]* 50
for idx, ch in enumerate(montage_name): 
    view4.by_label_params[f'label{idx}', 'color'] = colormap[idx]
    
view4.params['exclusive_mode'] = False
view4.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.add_view(view4, orientation='horizontal')

win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

## Display all spikes detected by SpikeDet (montage) 

Load .mat file containing detected spike

In [ ]:
mat = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/spikes_spmeeg_{file}_repaired_montage.mat", squeeze_me=True, struct_as_record=False)
data = mat['D'].trials.events
result = {}
for s in data:
    times = np.asarray(s.time).flatten()
    ch_name = str(np.asarray(s.value).flatten()[0]).replace("EEG_", "")
    if ch_name not in result:
        result[ch_name] = []
    result[ch_name].extend(times.tolist())
ordered_result = {k: result[k] for k in montage_name if k in result}

scatter_indexes = {
    i: pd.Series(np.asarray(v).ravel() * samplerate, name="start time", dtype=int)
    for i, v in enumerate(ordered_result.values())
}

order_index = {k: i for i, k in enumerate(montage_name)}

scatter_channels = {
    idx: [order_index[k]]
    for idx, k in enumerate(ordered_result.keys())
    if k in order_index
}

With sleep scoring + spikes (montage)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(sts_file).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels, scatter_colors= {i: "#FF000084" for i in range(len(montage_name))},channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'


## Display spikes detected by SpikeDet (montage) + Global clustering  

In [ ]:
matsp = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/sp_spikes_spmeeg_{file}_repaired_montage.mat", squeeze_me=True, struct_as_record=False)
GlobalSpike = matsp['GlobalSpikes']

clusters = np.array(GlobalSpike.Clusters.ClusterIndices)    
nb_cluster = int(clusters.max())
nb_cluster_tot = nb_cluster * len(montage_name_eeg)
print(f"{nb_cluster} clusters found")

scatter_channels = defaultdict(list)
scatter_indexes = defaultdict(list)

for clus in np.arange(0, nb_cluster_tot, 1):
    scatter_channels[clus+1]=[np.repeat(np.arange(0,len(montage_name_eeg)), nb_cluster)[clus]]
    scatter_indexes[clus+1]=[]

for spike in np.arange(0, len(clusters), 1):    
    key = int(clusters[spike] + (GlobalSpike.GenDet.SpikesInChannel[spike] * nb_cluster)- nb_cluster)
    scatter_indexes[key].append(GlobalSpike.GenDet.Epoch[spike])
    
scatter_channels = dict(sorted(scatter_channels.items()))
scatter_indexes = {k: pd.Series(v) for k, v in scatter_indexes.items()}
scatter_indexes = dict(sorted(scatter_indexes.items()))

groups = {}
for k, v in scatter_channels.items():
    groups.setdefault(v[0], []).append(k)
color_dict = {}
cmap = matplotlib.colormaps['rainbow']
for group_keys in groups.values():
    n = len(group_keys)
    norm = mcolors.Normalize(vmin=0, vmax=n-1)
    for i, key in enumerate(group_keys):
        color_dict[key] = mcolors.to_hex(cmap(norm(i))) + "80"

scatter_indexesO = scatter_indexes.copy()
scatter_channelsO = scatter_channels.copy()
color_dictO = color_dict.copy()


Remove spikes detected during Wake or Unstaged

In [ ]:
def find_stage(time_value, df):
    df['end_time'] = df['time'] + df['duration']
    row = df[(df['time'] <= time_value) & (time_value < df['end_time'])]
    if len(row) == 0:
        return None
    return row.iloc[0]['label']

stage_dict = {}
for key, time_list in scatter_indexesO.items():
    stage_dict[key] = [find_stage(t, SleepScoring) for t in time_list/samplerate]
    
labels_to_remove = {None, 'WAKE'}
scatter_indexes = {}
for key in scatter_indexesO:
    times  = scatter_indexesO[key]
    stages = stage_dict[key]
    filtered_times = [
        t for t, s in zip(times, stages)
        if s not in labels_to_remove
    ]    
    scatter_indexes[key] = filtered_times
scatter_indexes = {k: pd.Series(v) for k, v in scatter_indexes.items()}
scatter_indexes = dict(sorted(scatter_indexes.items()))
scatter_indexesO=scatter_indexes.copy()

Select a cluster (optional)

In [ ]:
sel_cluster = 2  # 1 = purple, 2 = blue, 3 =  green, 4 = orange, 5 = red
sel_clusters= np.arange(sel_cluster, nb_cluster_tot, nb_cluster)
scatter_indexes = {k: v for k, v in scatter_indexesO.items() if k in sel_clusters}
scatter_channels= {k: v for k, v in scatter_channelsO.items() if k in sel_clusters}
color_dict= {k: v for k, v in color_dictO.items() if k in sel_clusters}
print(f"Cluster {sel_cluster} contains {sum(len(v) for v in scatter_indexes.values())} detected spikes")

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(sts_file).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels,  scatter_colors = color_dict, channel_names=montage_name)
view1 = TraceViewer(source=source)
#scatter_colors = color_dict,
view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'


## Display spikes detected by SpikeDet (montage) + Clustering per channel

Load .mat file containing cluster identity of detected spike

In [ ]:
matsp = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/sp_spikes_spmeeg_{file}_repaired_montage.mat", squeeze_me=True, struct_as_record=False)

sp_array = matsp['sp']
montage_name_eeg=montage_name[:-2]
SpikeClusters = {}
rand_val = 0
scatter_channels = {}
epoch_dict={}
for i, channel in enumerate(montage_name_eeg):
    clusters = np.array(sp_array[i].Clusters.ClusterIndices)    
    max_cluster = int(clusters.max())
    SpikeClusters[channel] = (clusters + rand_val).tolist()
    epoch_dict[channel] = np.array(sp_array[i].GenDet.Epoch)
    for j in range(rand_val, rand_val + max_cluster + 1):
        scatter_channels[j] = [i]
    rand_val += max_cluster + 1
    
scatter_indexes = {}
temp_dict = defaultdict(list)
for key in SpikeClusters:
    vals_A = SpikeClusters[key]
    vals_B = np.asarray(epoch_dict[key]).ravel()
    vals_B = np.floor(vals_B).astype(int)
    for a_val, b_val in zip(vals_A, vals_B):
        temp_dict[a_val].append(b_val)
scatter_indexes = dict(temp_dict)
scatter_indexes = {k: pd.Series(v) for k, v in scatter_indexes.items()}
scatter_indexes = dict(sorted(scatter_indexes.items()))

groups = {}
for k, v in scatter_channels.items():
    groups.setdefault(v[0], []).append(k)
color_dict = {}
cmap = matplotlib.colormaps['rainbow']
for group_keys in groups.values():
    n = len(group_keys)
    norm = mcolors.Normalize(vmin=0, vmax=n-1)
    for i, key in enumerate(group_keys):
        color_dict[key] = mcolors.to_hex(cmap(norm(i))) + "80"

Select a cluster in a channel (optional)

In [ ]:
sel_cluster = 8
scatter_indexes= {k: v for k, v in scatter_indexes.items() if k == sel_cluster}
scatter_channels= {k: v for k, v in scatter_channels.items() if k == sel_cluster}
color_dict= {k: v for k, v in color_dict.items() if k == sel_cluster}

With sleep scoring + spikes & cluster ID (montage)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(sts_file).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels,  scatter_colors = color_dict, channel_names=montage_name)
view1 = TraceViewer(source=source)
#scatter_colors = color_dict,
view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'


## Display spikes detected by Snooz (montage)

In [ ]:
events_dict = defaultdict(list)
with open(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}_repaired_montage.tsv", newline='') as f:
    reader = csv.DictReader(f, delimiter="\t", quotechar='"')
    for row in reader:
        start = float(row['start_sec'])
        channels = ast.literal_eval(row['channels'])  # Convert string list to Python list
        for ch in channels:
            events_dict[ch].append(start)

# Convert defaultdict to normal dict
events_dict = dict(events_dict)
print(events_dict)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

scatter_indexes = {
    i: pd.Series(np.asarray(v).ravel() * samplerate, name="start time", dtype=int)
    for i, v in enumerate(events_dict.values())
}

order_index = {k: i for i, k in enumerate(montage_name)}

scatter_channels = {
    idx: [order_index[k]]
    for idx, k in enumerate(events_dict.keys())
    if k in order_index
}

SleepScoring_filename = f'{Path(sts_file).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels, scatter_colors= {i: "#FF000084" for i in range(len(montage_name))},channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'


## Display spikes detected by SpikeDect (single channel) 

Load .mat file

In [ ]:
mat = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/spikes_spmeeg_{file}_repaired.mat", squeeze_me=True, struct_as_record=False)
data = mat['D'].trials.events
result = {}
for s in data:
    times = np.asarray(s.time).flatten()
    value_name = str(np.asarray(s.value).flatten()[0]).replace("EEG_", "")

    if value_name not in result:
        result[value_name] = []
    result[value_name].extend(times.tolist())
ordered_result = {k: result[k] for k in channels_clean if k in result}

With sleep scoring + spikes (no montage)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

scatter_indexes = {
    i: pd.Series(np.asarray(v).ravel() * samplerate, name="start time", dtype=int)
    for i, v in enumerate(ordered_result.values())
}

order_index = {k: i for i, k in enumerate(channels_clean)}

scatter_channels = {
    idx: [order_index[k]]
    for idx, k in enumerate(ordered_result.keys())
    if k in order_index
}

SleepScoring_filename = f'{Path(sts_file).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(eeg, samplerate, t_start, scatter_indexes, scatter_channels, channel_names=eeg_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(eeg_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'
